In [4]:
import os
import re
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from settings import LOGS_DIR,ANALYSIS_DIR


In [5]:
try:
    from xgboost import XGBRegressor
    xgb_available = True
except Exception:
    xgb_available = False

In [6]:
# Loads

OUT_DIR = ANALYSIS_DIR

os.makedirs(OUT_DIR, exist_ok=True)

def find_csv_files(logs_dir):
    csv_pattern = os.path.join(logs_dir, "metrics_log_Hybrid_10_*_*_*_*.csv")
    csv_files = glob.glob(csv_pattern)
    return sorted(csv_files)

def infer_batch_from_filename(fname):
    name = os.path.basename(fname)
    m = re.search(r"batch[_-]?(\d+)", name, re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r"msgs?[_-]?per[_-]?node[_-]?(\d+)", name, re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r"(\d+)(?=[^0-9]*\.csv$)", name)
    if m:
        return int(m.group(1))
    return None

def load_and_merge(files):
    rows = []
    per_batch_aggregates = []
    for f in files:
        df = pd.read_csv(f)
        batch = None
        if "batch_size" in df.columns:
            batch = int(df["batch_size"].iloc[0])
        else:
            batch = infer_batch_from_filename(f)
        if batch is None:
            if set(["batch_size","avg_latency","avg_energy"]).intersection(df.columns):
                agg = df.copy()
                per_batch_aggregates.append(agg)
                continue
            batch = -1
        if set(["node_id","latency_ms","bytes","battery","energy"]).issubset(df.columns):
            df2 = df[["node_id","latency_ms","bytes","battery","energy"]].copy()
            df2["batch_size"] = batch
            rows.append(df2)
        else:
            if "avg_latency" in df.columns and "avg_energy" in df.columns and "batch_size" in df.columns:
                per_batch_aggregates.append(df.copy())
            else:
                try:
                    df2 = df.copy()
                    df2["batch_size"] = batch
                    rows.append(df2)
                except Exception:
                    continue
    if rows:
        merged_nodes = pd.concat(rows, ignore_index=True)
    else:
        merged_nodes = pd.DataFrame()
    if per_batch_aggregates:
        merged_batches = pd.concat(per_batch_aggregates, ignore_index=True)
    else:
        merged_batches = pd.DataFrame()
    return merged_nodes, merged_batches

In [4]:
# preprocess
def build_communication_cost(df, latency_col="latency_ms", energy_col="energy"):
    df = df.copy()
    lat = df[latency_col].astype(float)
    en = df[energy_col].astype(float)
    scaler = MinMaxScaler()
    arr = np.vstack([lat.values, en.values]).T
    scaled = scaler.fit_transform(arr)
    df["norm_latency"] = scaled[:,0]
    df["norm_energy"] = scaled[:,1]
    df["communication_cost"] = df["norm_latency"] + df["norm_energy"]
    return df, scaler

def preprocess(df):
    df = df.copy()
    if "node_id" in df.columns:
        df["node_id"] = df["node_id"].astype(str)
        df = pd.get_dummies(df, columns=["node_id"], drop_first=True)
    features = [c for c in df.columns if c not in ["latency_ms","energy","communication_cost","norm_latency","norm_energy","avg_latency","avg_energy","std_latency","std_energy","min_latency","max_latency","min_energy","max_energy","total_energy","avg_battery","min_battery","node_count","batch_size"]]
    candidates = ["bytes","battery","batch_size"]
    for c in candidates:
        if c in df.columns and c not in features:
            features.append(c)
    if "batch_size" in df.columns:
        features.insert(0,"batch_size")
    features = [f for f in features if f in df.columns]
    X = df[features].astype(float)
    return X, df



In [5]:
# Train
def train_and_evaluate(X, y, model, name, out_dir=OUT_DIR):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True
    )
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    y_test = y_test.reset_index(drop=True)
    preds = pd.Series(preds)
    joblib.dump({"model": model, "scaler": scaler}, os.path.join(out_dir, f"{name}_model.pkl"))
    return {
        "mse": mean_squared_error(y_test, preds),
        "mae": mean_absolute_error(y_test, preds),
        "r2": r2_score(y_test, preds),
    }, y_test, preds, X_test.reset_index(drop=True)



In [6]:
# Plots
def plot_and_save(df, X, y, name, preds=None, X_test=None, y_test=None, out_dir=OUT_DIR):
    plt.figure(figsize=(10,8))
    corr = df.select_dtypes(include=[np.number]).corr()
    sns.heatmap(corr, annot=True, fmt=".2f")
    plt.title("correlation heatmap")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"corr_heatmap_{name}.png"))
    plt.close()
    
    if preds is not None and y_test is not None:
        yt = pd.Series(y_test).reset_index(drop=True)
        pt = pd.Series(preds).reset_index(drop=True)
        plt.figure(figsize=(8,6))
        plt.scatter(yt, pt, alpha=0.6)
        plt.xlabel("true")
        plt.ylabel("pred")
        plt.title(f"true vs pred - {name}")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"true_pred_{name}.png"))
        plt.close()
        
    if "batch_size" in df.columns:
        plt.figure(figsize=(8,6))
        sns.boxplot(x=df["batch_size"], y=y)
        plt.title(f"boxplot {name}")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"boxplot_{name}.png"))
        plt.close()

def feature_importance_plot(model, X_columns, name, out_dir=OUT_DIR):
    if hasattr(model, "feature_importances_"):
        fi = model.feature_importances_
        idx = np.argsort(fi)[::-1]
        plt.figure(figsize=(8,6))
        sns.barplot(x=fi[idx], y=np.array(X_columns)[idx])
        plt.title(f"feature importance {name}")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"feature_importance_{name}.png"))
        plt.close()

In [7]:
# Run


def main():
    files = find_csv_files(LOGS_DIR)
    nodes_df, batches_df = load_and_merge(files)
    if nodes_df.empty and not batches_df.empty:
        if "batch_size" in batches_df.columns:
            nodes_df = batches_df.copy()
            if "avg_latency" in nodes_df.columns and "avg_energy" in nodes_df.columns:
                nodes_df = nodes_df.rename(columns={"avg_latency":"latency_ms","avg_energy":"energy"})
    if nodes_df.empty:
        print("no usable node-level data found")
        return

    if "batch_size" not in nodes_df.columns:
        nodes_df["batch_size"] = -1

    nodes_df, scaler_for_norm = build_communication_cost(nodes_df, latency_col="latency_ms", energy_col="energy")

    X_all_raw, nodes_df = preprocess(nodes_df)
    if X_all_raw.shape[0] < 10:
        print("not enough rows for robust ML; consider collecting more data")
    X_all = X_all_raw.copy()

    targets = {
        "latency_ms": nodes_df["latency_ms"].astype(float),
        "energy": nodes_df["energy"].astype(float),
        "communication_cost": nodes_df["communication_cost"].astype(float)
    }

    results = {}
    for tgt_name, y in targets.items():
        X = X_all.copy()
        rf = RandomForestRegressor(n_estimators=200, random_state=42)
        rf_metrics, y_test, preds, X_test = train_and_evaluate(X, y, rf, f"rf_{tgt_name}")
        results[f"rf_{tgt_name}"] = rf_metrics
        plot_and_save(nodes_df, X, y, tgt_name, preds=preds, X_test=X_test)
        feature_importance_plot(rf, X.columns, f"rf_{tgt_name}")

        if xgb_available:
            xgb = XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
            xgb_metrics, y_test2, preds2, X_test2 = train_and_evaluate(X, y, xgb, f"xgb_{tgt_name}")
            results[f"xgb_{tgt_name}"] = xgb_metrics
            plot_and_save(nodes_df, X, y, tgt_name+"_xgb", preds=preds2, X_test=X_test2)
            feature_importance_plot(xgb, X.columns, f"xgb_{tgt_name}")

        mlp = MLPRegressor(hidden_layer_sizes=(128,64), max_iter=1000, random_state=42,early_stopping=True,validation_fraction=0.1)
        mlp_metrics, y_test3, preds3, X_test3 = train_and_evaluate(X, y, mlp, f"mlp_{tgt_name}")
        results[f"mlp_{tgt_name}"] = mlp_metrics
        plot_and_save(nodes_df, X, y, tgt_name+"_mlp", preds=preds3, X_test=X_test3)

    pd.DataFrame(results).T.to_csv(os.path.join(OUT_DIR, "model_metrics.csv"))
    joblib.dump(scaler_for_norm, os.path.join(OUT_DIR, "norm_scaler.pkl"))
    nodes_df.to_csv(os.path.join(OUT_DIR, "merged_nodes_with_cost.csv"), index=False)
    print("done. outputs in", OUT_DIR)

    

In [8]:
main()

done. outputs in /Users/amirosein/Desktop/project/PythonProjects/srb/thesis/main/output/analysis
